### 1. Setting Up OpenFace in Google Colab

In [ ]:
!pip install -qq gdown
!gdown "https://drive.google.com/uc?id=1rVuzyC_A7NiD5YNWPA2SqGOLNPOWusiF" -O OpenFace_Portable.tar.gz
!tar -xzf OpenFace_Portable.tar.gz
!chmod +x /content/OpenFace/build/bin/FaceLandmarkVidMulti
!apt-get update -qq && apt-get install -y -qq ffmpeg libopencv-dev libboost-all-dev
!pip install rarfile


Downloading...
From (original): https://drive.google.com/uc?id=1rVuzyC_A7NiD5YNWPA2SqGOLNPOWusiF
From (redirected): https://drive.google.com/uc?id=1rVuzyC_A7NiD5YNWPA2SqGOLNPOWusiF&confirm=t&uuid=6571d865-e562-4678-b132-fd43ef0f9838
To: /content/OpenFace_Portable.tar.gz
100% 6.51G/6.51G [01:39<00:00, 65.3MB/s]


### 2. Downloading and extracting the videos

In [ ]:
import requests
import rarfile
from pathlib import Path

dataset_url = "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar"
base_dir = Path.cwd()

filename = Path(dataset_url).name
rar_path = base_dir / filename

# Downloading the dataset
with requests.get(dataset_url, stream=True, verify=False) as response:
   response.raise_for_status()
   with open(rar_path, 'wb') as file:
      for chunk in response.iter_content(chunk_size=8192):
         file.write(chunk)

# Extracting the dataset
with rarfile.RarFile(rar_path) as rf:
   rf.extractall(path=base_dir)

rar_path.unlink()

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.crcv.ucf.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### 3. Running FaceLandmarkVidMulti on the "_c01" videos

In [ ]:
import subprocess

video_path = base_dir / "UCF-101"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

videos = [f for f in video_path.rglob('*.avi') if "_c01" in f.name]

total_videos = len(videos)
print(f"Processing {total_videos} videos...\n\n")

# Process each video in batches so we don't run out of memory
batch_size = 50
for i in range(0, total_videos, batch_size):
   batch = videos[i : i + batch_size]
   batch_num = (i // batch_size) + 1
   total_batches = (total_videos // batch_size) + 1
   print(f"Executing batch {batch_num} of {total_batches}...\n\n")

   openface_cmd = ["/content/OpenFace/build/bin/FaceLandmarkVidMulti"]

   for video in batch:
      openface_cmd.extend(["-f", str(video)])

   openface_cmd.extend(["-out_dir", str(output_dir)])

   process = subprocess.Popen(
      openface_cmd,
      stdout=subprocess.PIPE,
      stderr=subprocess.STDOUT,
      text=True
   )

   for line in process.stdout:
      print(line, end="")

   process.wait()

A streamkimeneten csak az utolsó 5000 sor látható.
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
90% Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
Face too small for landmark detection
100% 
Closing output recorder
Closing input reader
Closed successfully
Attempting to read from file: /content/UCF-101/YoYo/v_YoYo_g01_c01.avi
Devic

### 4. Calculating statistics

In [ ]:
import pandas as pd
from pathlib import Path

csv_files = list(output_dir.rglob("*.csv"))

csv_stems = {f.stem for f in output_dir.rglob("*.csv")}
failed_videos = [v.stem for v in videos if v.stem not in csv_stems]

results = []

# Calculate statistics per video
for file_path in csv_files:
    df = pd.read_csv(file_path, skipinitialspace=True)

    if not df.empty:
        stats = (
            df.groupby("face_id", as_index=False)["confidence"]
            .mean()
            .rename(columns={"confidence": "mean_confidence"})
        )

        stats["video_name"] = file_path.stem

        number_of_faces = df["face_id"].nunique()
        stats["number_of_faces"] = number_of_faces

        results.append(stats)

    if df.empty or df["success"].sum() == 0:
        failed_videos.append(file_path.stem)


report = pd.concat(results, ignore_index=True)
report = report[["video_name", "face_id", "mean_confidence", "number_of_faces"]]
report.to_csv("/content/stats_per_video.csv", index=False)

successful_videos = [v.stem for v in videos if v.stem not in failed_videos]

failed_path = "/content/failed_videos.txt"

with open(failed_path, 'w') as f:
    f.write(f"Total of {len(failed_videos)} failed videos.\n\n")
    for video in failed_videos:
        f.write(f"{video}\n")

success_path = "/content/successful_videos.txt"

with open(success_path, 'w') as f:
    f.write(f"Total of {len(successful_videos)} successful videos.\n\n")
    for video in successful_videos:
        f.write(f"{video}\n")